In [2]:
# Remove unwanted warning
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
#warnings.simplefilter(action='ignore', catgeory=RuntimeWarning)

import os

# Data Management
import pandas as pd
import numpy as np
import polars as pl
import pyarrow as pa
from pandas_datareader.data import DataReader
from ta import add_all_ta_features

# Statistics
from statsmodels.tsa.stattools import adfuller

# Unsupervised Machine Learning
from sklearn.decomposition import PCA

# Supervised Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

# Reporting
import matplotlib.pyplot as plt

In [3]:
# Global Variables
CSV_FILENAME = "stocks.csv"
PARQET_FILENAME = "stocks.parquet"
WORKING_DIR = "data/"
FEATURES = ["gvkey"]

### Data Extraction

In [4]:
if not os.path.exists(os.path.join(
        WORKING_DIR, CSV_FILENAME
    )):
    # read sample data
    file_path = os.path.join(
        WORKING_DIR, "ret_sample.csv"
    )
    raw = pl.read_csv(file_path)
    raw = raw.filter(pl.col("excntry").is_in(["CAN","USA"]))
    raw.write_csv(os.path.join(WORKING_DIR, CSV_FILENAME))

if not os.path.exists(PARQET_FILENAME):
    raw = pd.read_csv(os.path.join(WORKING_DIR, CSV_FILENAME), dtype={4: str})
    raw.to_parquet(PARQET_FILENAME, index=False, compression="snappy")


raw = pd.read_parquet(PARQET_FILENAME)




In [22]:

# Display basic information
print("DataFrame Head:")
print(raw.head())

DataFrame Head:
                id      date   ret_eom   gvkey  iid excntry  stock_ret  year  \
0  comp_001081_01C  20050228  20050228  1081.0  01C     CAN  -0.143457  2005   
1  comp_001096_01C  20050228  20050228  1096.0  01C     CAN   0.028077  2005   
2   comp_001117_02  20050228  20050228  1117.0   02     USA  -0.168627  2005   
3  comp_001186_01C  20050228  20050228  1186.0  01C     CAN   0.149056  2005   
4  comp_001243_01C  20050228  20050228  1243.0  01C     CAN   0.006239  2005   

   month  char_date  char_eom            me        prc  market_equity  \
0      2   20050131  20050131   2398.152284   5.448179    2398.152284   
1      2   20050131  20050131    301.116426  21.389148     301.116426   
2      2   20050131  20050131     32.808300   2.550000      32.808300   
3      2   20050131  20050131   1099.753789  12.776989    1099.753789   
4      2   20050131  20050131  14740.873131  39.945243   14740.873131   

   div12m_me  chcsho_12m  eqnpo_12m   ret_1_0   ret_3_1   ret_6_

In [6]:

print("\nDataFrame Info:")
raw.info()



DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1398807 entries, 0 to 1398806
Columns: 159 entries, id to qmj_safety
dtypes: float64(149), int64(7), object(3)
memory usage: 1.7+ GB


In [7]:

print("\nDataFrame Description:")
print(raw.describe(include='all'))


DataFrame Description:
                     id          date       ret_eom         gvkey      iid  \
count           1398807  1.398807e+06  1.398807e+06  1.398356e+06  1398356   
unique            16739           NaN           NaN           NaN       21   
top     comp_106995_01C           NaN           NaN           NaN       01   
freq                245           NaN           NaN           NaN  1021652   
mean                NaN  2.014337e+07  2.014337e+07  7.689555e+04      NaN   
std                 NaN  5.967078e+04  5.967080e+04  6.927447e+04      NaN   
min                 NaN  2.005020e+07  2.005023e+07  1.004000e+03      NaN   
25%                 NaN  2.009063e+07  2.009063e+07  1.826400e+04      NaN   
50%                 NaN  2.014073e+07  2.014073e+07  3.932500e+04      NaN   
75%                 NaN  2.020013e+07  2.020013e+07  1.434210e+05      NaN   
max                 NaN  2.025063e+07  2.025063e+07  3.562890e+05      NaN   

        excntry     stock_ret          

In [21]:
pd.set_option('display.max_columns', None)
raw

,id,date,ret_eom,gvkey,iid,excntry,stock_ret,year,month,char_date,char_eom,me,prc,market_equity,div12m_me,chcsho_12m,eqnpo_12m,ret_1_0,ret_3_1,ret_6_1,ret_9_1,ret_12_1,ret_12_7,ret_60_12,seas_1_1an,seas_1_1na,seas_2_5an,seas_2_5na,at_gr1,sale_gr1,capx_gr1,inv_gr1,debt_gr3,sale_gr3,capx_gr3,inv_gr1a,lti_gr1a,sti_gr1a,coa_gr1a,col_gr1a,cowc_gr1a,ncoa_gr1a,ncol_gr1a,nncoa_gr1a,fnl_gr1a,nfna_gr1a,tax_gr1a,be_gr1a,ebit_sale,gp_at,cop_at,ope_be,ni_be,ebit_bev,netis_at,eqnetis_at,dbnetis_at,oaccruals_at,oaccruals_ni,taccruals_at,taccruals_ni,noa_at,opex_at,at_turnover,sale_bev,rd_sale,cash_at,sale_emp_gr1,emp_gr1,ni_inc8q,noa_gr1a,ppeinv_gr1a,lnoa_gr1a,capx_gr2,saleq_gr1,niq_be,niq_at,niq_be_chg1,niq_at_chg1,rd5_at,dsale_dinv,dsale_drec,dgp_dsale,dsale_dsga,saleq_su,niq_su,capex_abn,op_atl1,gp_atl1,ope_bel1,cop_atl1,pi_nix,ocf_at,op_at,ocf_at_chg1,at_be,ocfq_saleq_std,tangibility,earnings_variability,aliq_at,f_score,o_score,z_score,intrinsic_value,kz_index,ni_ar1,ni_ivol,at_me,be_me,debt_me,netdebt_me,sale_me,ni_me,ocf_me,fcf_me,eqpo_me,eqnpo_me,rd_me,bev_mev,ebitda_mev,aliq_mat,eq_dur,beta_60m,resff3_12_1,resff3_6_1,mispricing_mgmt,mispricing_perf,zero_trades_21d,dolvol_126d,dolvol_var_126d,turnover_126d,turnover_var_126d,zero_trades_126d,zero_trades_252d,bidaskhl_21d,ivol_capm_21d,iskew_capm_21d,coskew_21d,beta_dimson_21d,ivol_ff3_21d,iskew_ff3_21d,ivol_hxz4_21d,iskew_hxz4_21d,rmax5_21d,rmax1_21d,rvol_21d,rskew_21d,ami_126d,ivol_capm_252d,betadown_252d,prc_highprc_252d,corr_1260d,betabab_1260d,rmax5_rvol_21d,age,qmj,qmj_prof,qmj_growth,qmj_safety
0,comp_001081_01C,20050228,20050228,1081.0,01C,CAN,-0.143457,2005,2,20050131,20050131,2398.152284,5.448179,2398.152284,0.013995,0.000398,0.010974,-0.207906,0.153639,0.053557,-0.006092,-0.090374,-0.104220,-0.317399,0.034255,-0.028417,-0.088488,0.005070,0.049429,0.086579,0.687557,0.024745,0.063371,0.003833,-0.201533,0.001835,0.002224,0.000000,0.020670,0.003490,0.017179,0.018794,1.188120e-02,0.006913,0.031499,-0.031499,0.002574,0.029607,0.006310,0.092638,0.068317,0.060825,-0.020637,0.004108,-0.071610,0.000008,-0.071618,-0.008583,-50.917087,-0.040082,-237.778875,0.830420,0.455388,0.527547,0.650978,NaN,0.018005,NaN,NaN,0.0,0.025283,0.188401,0.021149,0.806366,0.116643,0.051041,0.018483,0.069298,0.025362,NaN,0.040951,-0.028766,-0.241907,0.286038,0.250790,1.860809,0.156138,0.062687,0.097217,0.065970,0.071694,0.117318,0.000749,0.059734,-0.021259,2.634434,0.136370,0.566473,0.414883,0.471998,4.0,-1.410147,0.907301,1219.144018,1.795771,-0.747150,0.013934,3.305473,1.254719,1.686110,1.626597,1.702724,-0.025893,0.002477,-0.112447,0.024800,0.019304,NaN,0.995829,0.075173,0.436383,15.260682,0.650452,-0.205451,0.034675,0.557652,0.435362,0.002815,1.547222e+07,0.691406,0.005768,0.732769,0.002012,0.002301,0.004671,0.020146,0.488188,-0.102866,-0.432624,0.018152,1.251482,0.019123,0.591514,0.015727,0.045644,0.021520,0.807346,0.001373,0.017210,0.779315,0.672204,0.387781,0.845865,0.805580,541,-1.508294,-0.994164,-0.832048,-1.017248
1,comp_001096_01C,20050228,20050228,1096.0,01C,CAN,0.028077,2005,2,20050131,20050131,301.116426,21.389148,301.116426,0.020393,-0.016831,0.040018,-0.017738,0.082960,0.287212,0.324699,0.262129,-0.007359,1.274552,0.037158,0.017431,0.001489,NaN,0.145403,0.146400,NaN,0.062489,0.768323,0.399888,NaN,0.002257,0.005478,NaN,0.012648,0.000767,0.011880,0.112555,3.321376e-02,0.079342,0.052235,-0.052235,0.014255,0.043772,0.338205,0.089504,0.114657,0.119896,-0.009665,0.056129,-0.104357,-0.000428,-0.103930,-0.046265,-9.369915,-0.098500,-19.948927,1.039370,0.082206,0.160804,0.165961,NaN,0.003652,NaN,NaN,0.0,0.104486,0.159915,0.065102,NaN,0.116452,0.007062,0.001693,-0.011413,-0.002408,NaN,0.042274,-0.303871,0.039769,-0.050259,-0.451925,-0.162791,NaN,0.078336,0.102519,0.146894,0.131328,NaN,0.043963,0.068392,0.020499,4.198937,0.230239,0.519181,0.608544,0.593111,3.0,-0.456683,0.676485,NaN,2.430729,0.376474,0.007200,6.350006,1.512289,3.618079,3.594891,0.956297,-0.014616,0.279167,-0.207008,0.035904,0.023477,NaN